In [19]:
# ==========================================
# 1. INSTALL & IMPORT LIBRARIES
# ==========================================
import os
import warnings
from math import sqrt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from pmdarima.arima import auto_arima

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

In [ ]:
# ==========================================
# 2. DOWNLOAD & PREPARE DATASET
# ==========================================
TICKER = 'BBNI.JK'
START_DATE = '2021-09-20'
END_DATE = '2024-09-20'
CSV_FILENAME = f'{TICKER}.csv'

# Download data if not locally available
if not os.path.exists(CSV_FILENAME):
    print(f"Downloading data for {TICKER}...")
    stock_data = yf.download(TICKER, start=START_DATE, end=END_DATE)
    stock_data.to_csv(CSV_FILENAME)

# Load dataset from local CSV
bbni = pd.read_csv(CSV_FILENAME, index_col=0, parse_dates=True)

# Flatten MultiIndex columns if generated by yfinance
if isinstance(bbni.columns, pd.MultiIndex):
    bbni.columns = bbni.columns.get_level_values(0)

# Select key OHLCV features and drop missing values
bbni = bbni[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()

df_close = bbni['Close'].copy()

# Plot historical stock closing price
plt.figure(figsize=(14, 6))
plt.plot(df_close, label='BBNI Closing Price', color='#1f77b4')
plt.title('BBNI.JK Stock Price History (2021 - 2024)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (IDR)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 3 & 4. RESET PIPELINE: PURE VALUES ARIMA
# ==========================================

# --- Step A: Clean columns and extract prices ---
if isinstance(bbni.columns, pd.MultiIndex):
    bbni.columns = bbni.columns.get_level_values(0)
bbni.columns = bbni.columns.str.strip()

# Force numeric transformation
clean_close = pd.to_numeric(bbni['Close'], errors='coerce').dropna()

# --- Step B: Convert to pure NumPy arrays (Removes date index conflicts) ---
# Stripping the index prevents statsmodels from crashing on weekend gaps
raw_values = clean_close.values

# --- Step C: Chronological Train / Test Split ---
train_size = int(len(raw_values) * 0.8)
train_data_raw = raw_values[:train_size]
test_data_raw = raw_values[train_size:]

print(f"Data verification - Total elements: {len(raw_values)}, Training steps: {len(train_data_raw)}")

# --- Step D: Train ARIMA Model ---
print("\n--- Training ARIMA Model ---")
model_arima = auto_arima(
    train_data_raw,   # Feeding clean numeric array
    seasonal=False,       
    trace=True,           
    suppress_warnings=True,
    error_action='ignore' 
)

print(model_arima.summary())

# --- Step E: Predict and Extract Component Data ---
# Forecast next 7 periods safely 
forecast_arima = model_arima.predict(n_periods=7)
print("\n--- ARIMA Forecast for Next 7 Days ---")
print(forecast_arima)

# Calculate residuals on training data for the LSTM to learn from later
fitted_arima = model_arima.predict_in_sample()
arima_residuals = train_data_raw - fitted_arima
print("\n--- Residuals calculated successfully! ---")

In [ ]:
# ==========================================
# 5. MODELLING: BIDIRECTIONAL LSTM
# ==========================================
print("\n--- Training BiLSTM Model ---")

# Feature Scaling
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df_close.values.reshape(-1, 1))

# Helper function to convert time series to supervised learning format
def create_supervised_data(data, look_back=1):
    X, Y = [], []
    for i in range(len(data) - look_back):
        X.append(data[i:(i + look_back), 0])
        Y.append(data[i + look_back, 0])
    return np.array(X), np.array(Y)

LOOK_BACK = 1
X, y = create_supervised_data(scaled_data, LOOK_BACK)

# Split dataset for LSTM training/testing
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Reshape input to 3D tensor [samples, time steps, features]
X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

# Define BiLSTM Model Architecture
model_lstm = Sequential([
    Bidirectional(LSTM(128, return_sequences=True), input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(32)),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1)
])

model_lstm.compile(optimizer='adam', loss='mean_squared_error')

# Train the network
history = model_lstm.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=0,
    shuffle=False
)

# Iterative 7-day multi-step forecasting
last_window = X_test[-1].copy()
forecast_lstm_scaled = []

for _ in range(7):
    pred = model_lstm.predict(last_window.reshape(1, 1, LOOK_BACK), verbose=0)
    forecast_lstm_scaled.append(pred[0, 0])
    # Update feature window for the next timestep
    last_window = np.array([[pred[0, 0]]])

# Inverse transform scaled predictions to original values
forecast_lstm = scaler.inverse_transform(np.array(forecast_lstm_scaled).reshape(-1, 1)).flatten()

In [ ]:
# ==========================================
# 6. MODEL EVALUATION
# ==========================================
actual_7_days = test_data.iloc[:7].values

# Evaluate ARIMA Metrics
mape_arima = mean_absolute_percentage_error(actual_7_days, forecast_arima)
rmse_arima = sqrt(mean_squared_error(actual_7_days, forecast_arima))

# Evaluate LSTM Metrics
mape_lstm = mean_absolute_percentage_error(actual_7_days, forecast_lstm)
rmse_lstm = sqrt(mean_squared_error(actual_7_days, forecast_lstm))

print("\n" + "="*40)
print("     MODEL PERFORMANCE EVALUATION (7 DAYS)")
print("="*40)
print(f"ARIMA  -> RMSE: {rmse_arima:.2f} | MAPE: {mape_arima*100:.2f}%")
print(f"BiLSTM -> RMSE: {rmse_lstm:.2f} | MAPE: {mape_lstm*100:.2f}%")
print("="*40)

In [ ]:
# ==========================================
# 7. FORECAST COMPARISON VISUALIZATION (FIXED)
# ==========================================

# 1. Re-fetch a clean, single-index series copy directly to guarantee proper dates
temp_download = yf.download('BBNI.JK', start='2021-09-20', end='2024-09-19', progress=False)

# 2. Extract valid DatetimeIndex and Price Values completely decoupled from 'Ticker' structures
original_dates = pd.to_datetime(temp_download.index)
actual_prices_slice = pd.to_numeric(temp_download['Close'].values.flatten(), errors='coerce')[-60:]
actual_dates_slice = original_dates[-60:]

# 3. Generate future business projection dates safely
future_dates = pd.date_range(start=original_dates[-1], periods=8, freq='B')[1:]

# 4. Standardize temporal dimensions into text-strings to avoid timezone crashes
actual_x_strings = actual_dates_slice.strftime('%Y-%m-%d')
future_x_strings = future_dates.strftime('%Y-%m-%d')

# 5. Initialize Graph Layout
plt.figure(figsize=(14, 6))

# 6. Plot clean chronological values
plt.plot(actual_x_strings, actual_prices_slice, label='Actual Price (Last 60 Days)', color='black', linewidth=1.5)

# 7. Plot model forecasting arrays 
plt.plot(future_x_strings, forecast_arima.flatten(), label='ARIMA 7-Day Forecast', color='red', linestyle='--', marker='o')
plt.plot(future_x_strings, forecast_lstm.flatten(), label='BiLSTM 7-Day Forecast', color='green', linestyle='--', marker='s')

# 8. Styling Adjustments
plt.title('7-Day Price Forecast Comparison: ARIMA vs BiLSTM', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Stock Price (IDR)', fontsize=12)

# Rotate labels and prune excessive ticks to maximize visibility space
plt.xticks(rotation=45, ha='right')
ax = plt.gca()
for i, label in enumerate(ax.get_xticklabels()):
    if i % 5 != 0 and i < len(actual_x_strings):
        label.set_visible(False)

plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()